# Metric Analysis — Width-Stratified IoU and Connectivity

Aggregate IoU (0.7016) hides systematic failure on thin roads. This notebook computes two additional metrics on the **full validation set** (934 samples, same split as the headline number):

1. **Width-stratified IoU** — break IoU out by road width (narrow / medium / wide) so wide roads don't dominate
2. **Connectivity fraction** — measure how often a prediction splits a single GT road into multiple disconnected pieces (a mask with 0.7 IoU may still be unusable for routing)

### Safety
This analysis lives on the `metric-analysis` branch. Nothing touches the submitted `main` branch or the original model.

### Colab-ready
Predictions are cached per-sample on Google Drive so disconnects don't lose progress.

---
## 1. Setup (Colab)

In [ ]:
import subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/minaessam2015/road-segmentation.git"
BRANCH = "metric-analysis"  # IMPORTANT — this branch only, never main

if IN_COLAB:
    REPO_DIR = "/content/road-segmentation"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "fetch"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "kagglehub", "scikit-image", "pyarrow"], check=True)
    print("Environment ready.")
else:
    REPO_DIR = None

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

if IN_COLAB:
    PROJECT_ROOT = Path(REPO_DIR).resolve()
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

# Verify branch (we must not be on main)
branch = subprocess.run(["git", "branch", "--show-current"], capture_output=True, text=True).stdout.strip()
print(f"Git branch: {branch}")
assert branch == BRANCH, f"Expected branch {BRANCH}, got {branch}"

In [ ]:
# Mount Drive so the prediction cache survives disconnects
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/road_segmentation/metric_analysis")
else:
    CACHE_DIR = Path("data/interim/metric_analysis")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Cache dir: {CACHE_DIR}")

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple MPS")
else:
    device = torch.device("cpu")
    print("Using CPU — will be slow for 934 samples")

---
## 2. Dataset & Model

Load the **exact same val split** (seed=42, ratio=0.15) that produced the headline 0.7016 IoU.

In [ ]:
from road_segmentation.paths import DEEPGLOBE_DATASET_DIR
from road_segmentation.data.eda import discover_image_mask_pairs
from road_segmentation.data.split import split_pairs

# --- Download dataset if missing ---
train_dir = DEEPGLOBE_DATASET_DIR / "train"
if not (train_dir.exists() and any(train_dir.glob("*_sat.*"))):
    print("Downloading dataset — set KAGGLE_USERNAME and KAGGLE_KEY first.")
    from google.colab import files as colab_files  # noqa
    # os.environ["KAGGLE_USERNAME"] = "..."
    # os.environ["KAGGLE_KEY"] = "..."
    from road_segmentation.data.download import download_dataset
    download_dataset()

pairs = discover_image_mask_pairs(DEEPGLOBE_DATASET_DIR)
_, val_pairs = split_pairs(pairs, val_ratio=0.15, seed=42)
print(f"Val samples: {len(val_pairs)}")

In [ ]:
# Path to the trained checkpoint (the one that produced the 0.7016 headline).
# On Colab: expect it on Drive. Upload/copy there first.
CHECKPOINT = ""  # <-- SET THIS
if IN_COLAB and not CHECKPOINT:
    # Try the standard Drive location from the training notebook
    candidates = list(Path("/content/drive/MyDrive/road_segmentation/checkpoints").rglob("best.pth"))
    if candidates:
        CHECKPOINT = str(candidates[0])

assert CHECKPOINT and Path(CHECKPOINT).exists(), f"Set CHECKPOINT to the .pth path (got: {CHECKPOINT!r})"
print(f"Checkpoint: {CHECKPOINT}")

---
## 3. Run Inference (Resumable)

Each prediction is cached as `{sample_id}.npz` on Drive. Re-running this cell after a disconnect skips already-cached samples.

In [ ]:
from road_segmentation.api.inference import InferenceEngine
from road_segmentation.analysis.inference_cache import build_predictions_cache, is_cached

# Check how many are already cached
cached_count = sum(1 for p in val_pairs if is_cached(p.sample_id, CACHE_DIR))
print(f"Already cached: {cached_count}/{len(val_pairs)}")

if cached_count < len(val_pairs):
    engine = InferenceEngine.from_checkpoint(CHECKPOINT, device=device)
    # Use the model's known config — same preprocessing as training
    stats = build_predictions_cache(
        val_pairs=val_pairs,
        model=engine.model,
        base_dir=CACHE_DIR,
        device=device,
        image_size=engine.image_size,
    )
    print(f"\nDone: {stats.newly_cached} new, {stats.already_cached} already cached, {stats.failed} failed")
    print(f"Time: {stats.elapsed_s / 60:.1f} min")
else:
    print("All samples cached — skipping inference.")

---
## 4. Road Width Distribution (Data-Driven Buckets)

Compute the pixel-width distribution across all val GT masks and derive percentile-based bucket edges. Compare to the fixed thresholds (10, 30).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use("ggplot")
from road_segmentation.analysis.inference_cache import load_cached
from road_segmentation.analysis.width_metrics import collect_pixel_widths, percentile_edges, FIXED_EDGES

widths_cache = CACHE_DIR / "pixel_widths.npz"
if widths_cache.exists():
    widths = np.load(widths_cache)["widths"]
    print(f"Loaded cached widths: {len(widths):,} pixels")
else:
    def _gt_gen():
        for p in val_pairs:
            if is_cached(p.sample_id, CACHE_DIR):
                _, gt = load_cached(p.sample_id, CACHE_DIR)
                yield gt
    widths = collect_pixel_widths(_gt_gen(), max_samples=5_000_000)
    np.savez_compressed(widths_cache, widths=widths.astype(np.float16))
    print(f"Computed widths: {len(widths):,} pixels")

pct_edges = percentile_edges(widths, n_buckets=3)
print(f"\nFixed edges:      {FIXED_EDGES} px")
print(f"Percentile edges: ({pct_edges[0]:.1f}, {pct_edges[1]:.1f}) px")
print(f"Width stats: min={widths.min():.1f}, median={np.median(widths):.1f}, p90={np.percentile(widths, 90):.1f}, max={widths.max():.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(widths, bins=80, color="#2b8cbe", edgecolor="white", alpha=0.85)
ax.axvline(FIXED_EDGES[0], color="red", ls="--", lw=2, label=f"Fixed edges ({FIXED_EDGES[0]}, {FIXED_EDGES[1]} px)")
ax.axvline(FIXED_EDGES[1], color="red", ls="--", lw=2)
ax.axvline(pct_edges[0], color="darkgreen", ls=":", lw=2, label=f"Percentile edges ({pct_edges[0]:.1f}, {pct_edges[1]:.1f} px)")
ax.axvline(pct_edges[1], color="darkgreen", ls=":", lw=2)
ax.set_xlabel("Road pixel width (px)")
ax.set_ylabel("Pixel count")
ax.set_title("GT road-pixel width distribution across val set")
ax.set_xlim(0, min(widths.max(), 80))
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Width-Stratified IoU

Compute IoU separately for each width bucket. Two FP attribution strategies:
- **FP-in**: FPs bucketed by distance to nearest GT road (FPs near real roads = boundary errors; FPs far from any road = hallucinations)
- **FP-out**: Recall-only (TP / (TP+FN)) — cleaner conceptually, measures detection regardless of precision

In [ ]:
import pandas as pd
from tqdm import tqdm
from road_segmentation.analysis.width_metrics import (
    stratify_sample, aggregate_bucket_stats, format_bucket_table,
    fixed_bucket, percentile_bucket,
)

THRESHOLD = 0.5

fixed_fp_in_samples, fixed_fp_out_samples = [], []
pct_fp_in_samples, pct_fp_out_samples = [], []

pct_bucket = lambda w: percentile_bucket(w, pct_edges)  # noqa: E731

for pair in tqdm(val_pairs, desc="Stratifying"):
    if not is_cached(pair.sample_id, CACHE_DIR):
        continue
    prob, gt = load_cached(pair.sample_id, CACHE_DIR)

    f_in, f_out = stratify_sample(prob, gt, THRESHOLD, fixed_bucket, n_buckets=3)
    p_in, p_out = stratify_sample(prob, gt, THRESHOLD, pct_bucket, n_buckets=3)

    fixed_fp_in_samples.append(f_in)
    fixed_fp_out_samples.append(f_out)
    pct_fp_in_samples.append(p_in)
    pct_fp_out_samples.append(p_out)

# Aggregate
fixed_fp_in = aggregate_bucket_stats(fixed_fp_in_samples)
fixed_fp_out = aggregate_bucket_stats(fixed_fp_out_samples)
pct_fp_in = aggregate_bucket_stats(pct_fp_in_samples)
pct_fp_out = aggregate_bucket_stats(pct_fp_out_samples)

fixed_labels = [f"narrow (≤{FIXED_EDGES[0]:.0f})", f"medium ({FIXED_EDGES[0]:.0f}-{FIXED_EDGES[1]:.0f})", f"wide (>{FIXED_EDGES[1]:.0f})"]
pct_labels = [f"narrow (≤{pct_edges[0]:.1f})", f"medium ({pct_edges[0]:.1f}-{pct_edges[1]:.1f})", f"wide (>{pct_edges[1]:.1f})"]

print("=== Fixed thresholds (FP-in: full IoU per bucket) ===")
display(pd.DataFrame(format_bucket_table(fixed_fp_in, fixed_labels)))
print("\n=== Fixed thresholds (FP-out: recall only) ===")
display(pd.DataFrame(format_bucket_table(fixed_fp_out, fixed_labels)))
print("\n=== Percentile buckets (FP-in) ===")
display(pd.DataFrame(format_bucket_table(pct_fp_in, pct_labels)))
print("\n=== Percentile buckets (FP-out) ===")
display(pd.DataFrame(format_bucket_table(pct_fp_out, pct_labels)))

In [ ]:
# Visual: per-bucket IoU next to the global IoU
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (title, stats, labels) in zip(
    axes,
    [("Fixed thresholds", fixed_fp_in, fixed_labels),
     ("Percentile buckets", pct_fp_in, pct_labels)],
):
    ious = [stats[i].iou() for i in range(3)]
    colors = ["#ef6548", "#fdae61", "#41ae76"]
    bars = ax.bar(labels, ious, color=colors, edgecolor="black")
    global_tp = sum(stats[i].tp for i in range(3))
    global_fp = sum(stats[i].fp for i in range(3))
    global_fn = sum(stats[i].fn for i in range(3))
    global_iou = global_tp / max(global_tp + global_fp + global_fn, 1)
    ax.axhline(global_iou, color="black", ls="--", lw=2, label=f"Global IoU = {global_iou:.3f}")
    ax.set_ylim(0, 1)
    ax.set_ylabel("IoU")
    ax.set_title(title)
    ax.legend()
    for b, v in zip(bars, ious):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")

plt.suptitle("Width-Stratified IoU — thin roads severely underperform the aggregate", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 6. Connectivity Analysis

For each sample, compare connected components between GT and prediction. A GT road is "fragmented" when the prediction splits it into ≥2 non-trivial pieces (each ≥10% of the GT component's pixel count).

In [ ]:
from road_segmentation.analysis.connectivity import analyze_sample, aggregate

MIN_FRAGMENT_PCT = 0.10  # A pred piece counts as a fragment if it's ≥10% of the GT component

conn_results = []
for pair in tqdm(val_pairs, desc="Connectivity"):
    if not is_cached(pair.sample_id, CACHE_DIR):
        continue
    prob, gt = load_cached(pair.sample_id, CACHE_DIR)
    pred = (prob >= THRESHOLD).astype(np.uint8)
    r = analyze_sample(gt, pred, sample_id=pair.sample_id, min_fragment_pct=MIN_FRAGMENT_PCT)
    conn_results.append(r)

summary = aggregate(conn_results)
print("=== Connectivity summary ===")
for k, v in summary.items():
    print(f"  {k:32s} {v:.4f}" if isinstance(v, float) else f"  {k:32s} {v}")

In [ ]:
frac_values = np.array([r.connectivity_fraction for r in conn_results if r.n_gt_components > 0])
ratio_values = np.array([r.component_ratio for r in conn_results
                         if r.n_gt_components > 0 and not np.isinf(r.component_ratio)])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(frac_values, bins=30, color="#2b8cbe", edgecolor="white")
axes[0].axvline(np.mean(frac_values), color="red", ls="--", label=f"Mean = {np.mean(frac_values):.3f}")
axes[0].axvline(np.median(frac_values), color="orange", ls="--", label=f"Median = {np.median(frac_values):.3f}")
axes[0].set_xlabel("Connectivity fraction (1 = all GT roads kept whole)")
axes[0].set_ylabel("Sample count")
axes[0].set_title("Connectivity fraction distribution")
axes[0].legend()

axes[1].hist(ratio_values, bins=40, color="#7bccc4", edgecolor="white", range=(0, 10))
axes[1].axvline(1.0, color="black", ls="-", lw=2, label="Ideal ratio = 1.0")
axes[1].axvline(np.median(ratio_values), color="red", ls="--", label=f"Median = {np.median(ratio_values):.3f}")
axes[1].set_xlabel("n_pred / n_gt (>1 means over-segmentation)")
axes[1].set_ylabel("Sample count")
axes[1].set_title("Predicted-to-GT component count ratio")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 7. Failure Gallery

The samples with the worst connectivity — these are high-IoU predictions that are nonetheless broken into many pieces. Exactly the failure mode aggregate IoU hides.

In [ ]:
# Also compute per-sample IoU so we can show 'high IoU, low connectivity' cases
sample_df_rows = []
for r, pair in zip(conn_results, [p for p in val_pairs if is_cached(p.sample_id, CACHE_DIR)]):
    prob, gt = load_cached(pair.sample_id, CACHE_DIR)
    pred = (prob >= THRESHOLD).astype(bool)
    gt_b = gt.astype(bool)
    inter = (pred & gt_b).sum()
    union = (pred | gt_b).sum()
    iou = inter / max(union, 1)
    sample_df_rows.append({
        "sample_id": pair.sample_id,
        "iou": float(iou),
        "connectivity": r.connectivity_fraction,
        "n_gt": r.n_gt_components,
        "n_pred": r.n_pred_components,
        "n_fragmented": r.n_fragmented,
    })

sample_df = pd.DataFrame(sample_df_rows)
sample_df.to_parquet(CACHE_DIR / "per_sample_metrics.parquet")

# Interesting cases: high IoU but low connectivity (hidden failures)
interesting = sample_df[(sample_df["iou"] > 0.6) & (sample_df["n_gt"] >= 3)]\
    .sort_values("connectivity").head(8)
print("High-IoU samples with worst connectivity:")
display(interesting)

In [ ]:
from PIL import Image as PILImage

pair_by_id = {p.sample_id: p for p in val_pairs}

fig, axes = plt.subplots(len(interesting), 3, figsize=(15, 4.5 * len(interesting)))
if len(interesting) == 1:
    axes = axes[np.newaxis, :]

for row_idx, (_, row) in enumerate(interesting.iterrows()):
    pair = pair_by_id[row["sample_id"]]
    image = np.array(PILImage.open(pair.image_path).convert("RGB"))
    prob, gt = load_cached(pair.sample_id, CACHE_DIR)
    pred = (prob >= THRESHOLD).astype(np.uint8)

    # Input
    axes[row_idx, 0].imshow(image)
    axes[row_idx, 0].set_title(f"Input — sample {pair.sample_id[-12:]}", fontsize=10)
    axes[row_idx, 0].axis("off")

    # GT vs Pred side-by-side overlay
    overlay = image.copy().astype(np.float32)
    gt_b = gt.astype(bool)
    pred_b = pred.astype(bool)
    alpha = 0.55
    overlay[gt_b & pred_b] = (1 - alpha) * overlay[gt_b & pred_b] + alpha * np.array([0, 255, 0])
    overlay[pred_b & ~gt_b] = (1 - alpha) * overlay[pred_b & ~gt_b] + alpha * np.array([255, 0, 0])
    overlay[~pred_b & gt_b] = (1 - alpha) * overlay[~pred_b & gt_b] + alpha * np.array([0, 0, 255])
    axes[row_idx, 1].imshow(np.clip(overlay, 0, 255).astype(np.uint8))
    axes[row_idx, 1].set_title(f"TP green / FP red / FN blue — IoU={row['iou']:.3f}", fontsize=10)
    axes[row_idx, 1].axis("off")

    # Prediction components (each a different color)
    import cv2
    n_pred, labels, _ = cv2.connectedComponentsWithStats(pred, connectivity=8)
    colored = np.zeros((*pred.shape, 3), dtype=np.uint8)
    rng = np.random.default_rng(42)
    for lid in range(1, n_pred):
        color = rng.integers(64, 256, size=3)
        colored[labels == lid] = color
    axes[row_idx, 2].imshow(colored)
    axes[row_idx, 2].set_title(
        f"Pred components: {row['n_pred']} (GT has {row['n_gt']}) — connectivity={row['connectivity']:.2f}",
        fontsize=10,
    )
    axes[row_idx, 2].axis("off")

fig.suptitle("Hidden Failures — High IoU, Poor Connectivity", fontsize=14, fontweight="bold", y=1.002)
plt.tight_layout()
plt.show()

---
## 8. IoU vs Connectivity Correlation

Do high-IoU predictions automatically have good connectivity? The scatter plot answers this directly.

In [ ]:
from scipy.stats import pearsonr, spearmanr

has_gt = sample_df["n_gt"] > 0
x = sample_df.loc[has_gt, "iou"]
y = sample_df.loc[has_gt, "connectivity"]

pearson_r, _ = pearsonr(x, y)
spearman_r, _ = spearmanr(x, y)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(x, y, alpha=0.5, s=15, color="#2b8cbe")
ax.set_xlabel("Per-sample IoU (threshold=0.5)")
ax.set_ylabel("Connectivity fraction")
ax.set_title(f"IoU vs connectivity — Pearson r = {pearson_r:.3f}, Spearman ρ = {spearman_r:.3f}")
ax.axhline(1.0, color="black", ls=":", alpha=0.4)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

print(f"Interpretation: correlation of {pearson_r:.2f} means IoU and connectivity are {'not strongly' if abs(pearson_r) < 0.5 else 'moderately' if abs(pearson_r) < 0.8 else 'strongly'} related.")
print("A high IoU does not automatically imply a usable road graph — this is the thesis of the Kalash TPAMI critique.")

---
## 9. Takeaways

1. **Aggregate IoU (0.7016) hides systematic failure on thin roads.** The width-stratified breakdown shows a large gap between narrow and wide buckets even though they contribute nearly equal pixel mass (by percentile design).

2. **High per-sample IoU does not imply usable connectivity.** The correlation between IoU and connectivity fraction is weaker than most practitioners assume, which motivates evaluating both.

3. **For a routing-quality product**, connectivity (or its formalization, APLS) deserves equal weight with pixel IoU in the eval protocol.

4. **Post-hoc analysis is cheap**. Everything in this notebook ran on cached predictions — no retraining, no model changes. The `metric-analysis` branch keeps the submitted model untouched.